# Notebook 03 - Feature Engineering
## Fake News Detection - NLP Assignment
### Person 2: W.A. Irusha Madushan (CIT-24-01-0514)

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pickle
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
# Load the cleaned dataset
df = pd.read_csv('../data/cleaned_data.csv')

# Drop any remaining null values in processed_text
df = df.dropna(subset=['processed_text'])

print("Dataset loaded successfully")
print("Shape:", df.shape)

Dataset loaded successfully
Shape: (72041, 5)


## 1. Train/Test Split

Before creating features, the dataset is split into training and 
testing sets. I used an 80/20 split which means 80% of the data 
is used for training and 20% is used for testing. This is a 
standard approach in machine learning projects.

In [3]:
X = df['processed_text']
y = df['label']

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train/Test Split Complete")
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("\nTraining label distribution:")
print(y_train.value_counts())
print("\nTesting label distribution:")
print(y_test.value_counts())

Train/Test Split Complete
Training samples: 57632
Testing samples: 14409

Training label distribution:
label
1    29610
0    28022
Name: count, dtype: int64

Testing label distribution:
label
1    7403
0    7006
Name: count, dtype: int64


## 2. TF-IDF Vectorization for Random Forest

TF-IDF stands for Term Frequency - Inverse Document Frequency. 
It converts text into numbers by giving higher scores to words 
that appear frequently in one article but not in many other 
articles. This makes it good for identifying words that are 
important for fake or real news classification.

I used a maximum of 50,000 features and included both single 
words and two word combinations (ngram_range=(1,2)) to capture 
more context from the articles.

In [4]:
# TF-IDF Vectorization for Random Forest
tfidf_vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

# Fit on training data and transform both train and test
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("TF-IDF Vectorization Complete")
print("Training matrix shape:", X_train_tfidf.shape)
print("Testing matrix shape:", X_test_tfidf.shape)
print("Total features (vocabulary size):", len(tfidf_vectorizer.vocabulary_))

TF-IDF Vectorization Complete
Training matrix shape: (57632, 50000)
Testing matrix shape: (14409, 50000)
Total features (vocabulary size): 50000


## 3. Saving TF-IDF Vectorizer

The TF-IDF vectorizer is saved so that it can be loaded and 
used again in the model training and evaluation notebooks 
without having to refit it on the data again.

In [6]:
import scipy.sparse as sp

# Save TF-IDF vectorizer
with open('../models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)

# Save as sparse matrix (memory efficient)
sp.save_npz('../models/X_train_tfidf.npz', X_train_tfidf)
sp.save_npz('../models/X_test_tfidf.npz', X_test_tfidf)

# Save labels
np.save('../models/y_train.npy', y_train.values)
np.save('../models/y_test.npy', y_test.values)

print("TF-IDF vectorizer saved successfully")
print("Train and test sparse matrices saved successfully")
print("Labels saved successfully")

TF-IDF vectorizer saved successfully
Train and test sparse matrices saved successfully
Labels saved successfully
